# Volume 3. Parser Failure Cases

Question: what happens when the toy parser is asked to handle inputs outside the tiny training story?

High-quality demos include failure surfaces. This notebook keeps the same three-word NEMO-style parser and probes swapped nouns, missing structure, duplicate tokens, and unknown words.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from neural_assemblies.assembly_calculus import NemoParser
from neural_assemblies.core.brain import Brain


In [ ]:
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
parser = NemoParser(brain, n=5_000, k=60, beta=0.10, rounds=6)
parser.setup_areas()
for word, category, grounding in [
    ("dog", "noun", "vis_dog"),
    ("chases", "verb", "mot_chases"),
    ("cat", "noun", "vis_cat"),
]:
    parser.register_word(word, category, grounding)
parser.train_lexicon()
parser.train_roles([["dog", "chases", "cat"]])
parser.train_word_order([["dog", "chases", "cat"]])


A small wrapper makes success and failure equally tabular. Notice that duplicate words expose a limitation of dictionary-shaped parse output.


In [ ]:
def safe_parse(label, words):
    try:
        result = parser.parse(words)
    except Exception as exc:
        return [{
            "case": label,
            "position": None,
            "word": " ".join(words),
            "category": None,
            "role": None,
            "status": type(exc).__name__,
            "note": str(exc),
        }]

    rows = []
    duplicate_tokens = len(set(words)) != len(words)
    for position, word in enumerate(words, start=1):
        rows.append({
            "case": label,
            "position": position,
            "word": word,
            "category": result["categories"].get(word),
            "role": result["roles"].get(word),
            "status": "ok",
            "note": "duplicate token shares one dict entry" if duplicate_tokens else "",
        })
    return rows

cases = {
    "trained sentence": ["dog", "chases", "cat"],
    "swapped nouns": ["cat", "chases", "dog"],
    "missing verb": ["dog", "cat"],
    "duplicate noun": ["dog", "chases", "dog"],
    "unknown word": ["fox", "chases", "dog"],
}
rows = []
for label, words in cases.items():
    rows.extend(safe_parse(label, words))
failures = pd.DataFrame(rows)
failures


In [ ]:
summary = failures.groupby(["case", "status"]).size().reset_index(name="rows")
summary


In [ ]:
role_counts = failures[failures["status"] == "ok"].groupby(["case", "role"]).size().unstack(fill_value=0)
ax = role_counts.plot(kind="bar", stacked=True, figsize=(7.2, 3.5), color=["#4d9de0", "#59a14f", "#e15554"])
ax.set_ylabel("token rows")
ax.set_title("Role assignments by probe case")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.xticks(rotation=30, ha="right")
plt.show()
plt.close(ax.figure)


These are not embarrassing edge cases; they are the point of the notebook. The parser API currently demonstrates category/role/order plumbing for a tiny controlled setup. Unknown-word handling, duplicate-token structure, and richer grammar need explicit future machinery.
